In [18]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score


In [19]:
df = pd.read_csv('users_behavior.csv')

In [20]:
# realizamos análisis exploratorio de los datos
print(df.info())
print()
print(df.head())
print()
print(df.describe())
print()
print(f'Valores nulos: {df.isnull().sum()}')
print()
print(f'Valores duplicados: {df.duplicated().sum()}')
print()
print(f'Shape: {df.shape}')



<class 'pandas.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB
None

   calls  minutes  messages   mb_used  is_ultra
0   40.0   311.90      83.0  19915.42         0
1   85.0   516.75      56.0  22696.96         0
2   77.0   467.66      86.0  21060.45         0
3  106.0   745.53      81.0   8437.39         1
4   66.0   418.74       1.0  14502.75         0

             calls      minutes     messages       mb_used     is_ultra
count  3214.000000  3214.000000  3214.000000   3214.000000  3214.000000
mean     63.038892   438.208787    38.281269  17207.673836     0.306472
std      33.236368   234.569872    36.148326   7570.968246     0.4611

In [21]:
# segmentamos los datos en features y target
features = df.drop(['is_ultra'], axis=1)
target = df['is_ultra']

# 60% entrenamiento / 40% resto
features_train, features_rest, target_train, target_rest = train_test_split(
    features, target, test_size=0.4, random_state=12345
)

# 40% resto -> 20% validación / 20% prueba
features_valid, features_test, target_valid, target_test = train_test_split(
    features_rest, target_rest, test_size=0.5, random_state=12345
)

print(f'Train: {features_train.shape}, Valid: {features_valid.shape}, Test: {features_test.shape}')


Train: (1928, 4), Valid: (643, 4), Test: (643, 4)


## Paso 3: Investigación de hiperparámetros

### Árbol de decisión

In [22]:
tree_best_score = 0
tree_best_depth = 0
for depth in range(1, 11):
    model = DecisionTreeClassifier(max_depth=depth, random_state=12345)
    model.fit(features_train, target_train)
    predictions = model.predict(features_valid)
    result = accuracy_score(target_valid, predictions)
    if result > tree_best_score:
        tree_best_score = result
        tree_best_depth = depth

print(f'max_depth={tree_best_depth}, exactitud en validación={tree_best_score}')


max_depth=3, exactitud en validación=0.7853810264385692


### Bosque aleatorio

In [23]:
forest_best_score = 0
forest_best_depth = 0
forest_best_est = 0
for depth in range(1, 11):
    for est in range(10, 51, 10):
        model = RandomForestClassifier(n_estimators=est, max_depth=depth, random_state=12345)
        model.fit(features_train, target_train)
        predictions = model.predict(features_valid)
        result = accuracy_score(target_valid, predictions)
        if result > forest_best_score:
            forest_best_score = result
            forest_best_depth = depth
            forest_best_est = est

print(f'n_estimators={forest_best_est}, max_depth={forest_best_depth}, exactitud en validación={forest_best_score}')


n_estimators=40, max_depth=8, exactitud en validación=0.8087091757387247


### Regresión logística

In [24]:
model = LogisticRegression(random_state=12345, solver='liblinear')
model.fit(features_train, target_train)
predictions = model.predict(features_valid)
logreg_score = accuracy_score(target_valid, predictions)
print(f'Regresión logística, exactitud en validación={logreg_score}')


Regresión logística, exactitud en validación=0.7573872472783826


In [25]:
resultados = pd.DataFrame({
    'modelo': ['Bosque aleatorio', 'Árbol de decisión', 'Regresión logística'],
    'exactitud_validacion': [forest_best_score, tree_best_score, logreg_score]
}).sort_values('exactitud_validacion', ascending=False).reset_index(drop=True)

resultados


,modelo,exactitud_validacion
0,Bosque aleatorio,0.808709
1,Árbol de decisión,0.785381
2,Regresión logística,0.757387
